# Eval, Merge& Deploy

**Module 9 · Lesson 9.6**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Prove It Beats the Base - Held-Out A/B + LLM-as-Judge
- The Benchmark Portfolio - Why MMLU Alone Lies
- LLM-as-Judge - Cheap, Fast, and Biased
- The Safety Gate - Fine-Tuning Can Break Alignment
- Merge the Adapter - LoRA Merge & Model Merging
- Quantize for the Target - GGUF, AWQ, FP8

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

## The Model That Looked Great Until It Shipped

> 💡 **Info**
>
> A team fine-tuned a support model. The training loss curve was *beautiful* - smooth, low, textbook. They shipped it. Two things they never checked: on held-out questions it actually scored **below the base model** (it had memorized phrasings, not learned the policy), and it had quietly lost most of its **safety refusals** - fine-tuning on benign support chats degraded its alignment. A low loss curve proves you fit the training set. It proves *nothing* about whether the model is better, or safe. This lesson is how you actually find out before your users do.

### 🎯 What you will be able to do after this lesson

- **Build** a held-out A/B evaluation that scores the fine-tuned model against the base with automated metrics + an LLM-as-judge (current google-genai SDK), and read a benchmark portfolio (MMLU-Pro / GPQA / IFEval / EvalPlus / LMArena).

- **Compare** merge strategies - LoRA `merge_and_unload` vs whole-model merging (linear / SLERP / TIES / DARE-TIES) - and when each applies.

- **Operate** the safety gate (HarmBench/ToxiGen + perplexity forgetting) and quantize to the target (GGUF Q4_K_M/Q6_K, AWQ+Marlin, FP8).

- **Evaluate** deployment: merge→Ollama vs vLLM vs vLLM multi-LoRA vs managed, and the India path (Sarvam, IndiaAI, Bhashini, DPDP).

> 📦 **Info**
>
> ✅ Before you startThis is the Module 9 closer - it takes the adapter you understood in **Lesson 9.5** and ships it. You should know what a LoRA adapter and merge are (9.5) and the QLoRA/serving basics (9.4). The deep production side - SLO monitoring, autoscaling, canary rollout, drift, cost engineering - is **Modules 12-14**; here we cover the eval, merge, quantize, and deploy-target decision.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 🏏 **Analogy**
>
> **Training is the tryout; evaluation is the match.** A bowler who is unplayable in the nets (training loss dropping) can still get hit around the park on match day (real user queries). You only learn the truth by playing an actual match - on unseen deliveries, against the incumbent, with the umpires (safety checks) watching. **Where it breaks down:** passing the match once is necessary but not sufficient - the season keeps going (production drift), and monitoring that is Modules 12-14.

> 📦 **Info**
>
> 🚫 Two misconceptions this lesson kills
> - **“Low training loss means a good model.”** It means you fit the training set. Only a held-out A/B against the base proves the model is actually better.
> - **“~87% on MMLU is a great score.”** MMLU is saturated - strong models all cluster near the top. It barely discriminates; use MMLU-Pro / GPQA / IFEval.

> 💡 **Info**
>
> ⚠️ Anti-patternShipping on the training-loss curve alone - no held-out A/B against the base, and no safety-regression gate. That is exactly how a model that looks great in training goes to production worse than the base *and* less safe than the base.

---

## 🎯 Concept 1: Prove It Beats the Base - Held-Out A/B + LLM-as-Judge

### Prove It Beats the Base - Held-Out A/B + LLM-as-Judge

Score the fine-tuned model against the base on unseen cases. Automated metrics AND an LLM judge on the dimensions that matter.

The first job is not “is the model good” - it is “is it *better than what I already had*?” So you always run an **A/B**: the same held-out questions through the base model and the fine-tuned model, scored the same way. For open-ended answers, an **LLM-as-judge** rates each response 0-1 on accuracy, helpfulness, and style. If the fine-tune does not beat the base, you do not ship it.

> ⚖️ **Analogy**
>
> It is a **blind taste test**. You do not ask “is this dish tasty?” in the abstract - you put the new recipe and the old one in identical bowls and have judges score both without knowing which is which. The only result that matters is the *difference*.

Your fine-tune's training loss is the lowest you have ever seen. What does that tell you about its quality for users?

**📝 `01_ab_eval.py`** - *runnable*

In [ ]:
# A/B EVAL - base vs fine-tuned on held-out cases (deterministic rule-judge stands in for the LLM).
def judge(response, expected_keywords):
    hits = sum(1 for k in expected_keywords if k.lower() in response.lower())
    return round(hits / len(expected_keywords), 3)

cases = [
    {"kw": ["7 days", "50%", "30"], "base": "Please check the website.",
     "ft": "Full refund within 7 days, 50% from 8-30 days, none after 30."},
    {"kw": ["14999", "lifetime"],   "base": "I am not sure of the price.",
     "ft": "It is 14999 rupees with lifetime access."},
    {"kw": ["python", "math"],      "base": "Various skills are needed.",
     "ft": "Python basics and high-school math."},
]
def eval_model(which):
    return round(sum(judge(c[which], c["kw"]) for c in cases) / len(cases), 3)

base, ft = eval_model("base"), eval_model("ft")
print(f"  base       score {base}")
print(f"  fine-tuned score {ft}   delta {ft-base:+.3f}")
print(f"  verdict: {'SHIP - fine-tune beats base' if ft > base else 'DO NOT SHIP - no gain over base'}")

# Output:
#   base       score 0.0
#   fine-tuned score 1.0   delta +1.000
#   verdict: SHIP - fine-tune beats base

- The rule-based `judge` stands in for an LLM judge so the block RUNS deterministically - it scores by how many expected facts a response contains.
- `eval_model` runs the SAME held-out cases through base and fine-tuned and averages the scores.
- The **delta** (fine-tuned minus base) is the whole point - a positive delta is your ship signal.
- In production you swap the rule-judge for a real LLM-as-judge - the next block shows the current SDK.

**📝 `01b_llm_judge.py`** - *google-genai*

> **Illustrative (not run here)** - needs `GOOGLE_API_KEY`. This is the real LLM-as-judge using the current google-genai SDK.

```python
# LLM-AS-JUDGE - the real thing (illustrative; needs an API key). CURRENT google-genai SDK.
from google import genai                      # the google-genai package (the old generativeai one is retired)
client = genai.Client()                        # reads GOOGLE_API_KEY

def judge(metric, question, expected, response):
    prompt = (f"Rate 0.0-1.0 the {metric} of this answer. Return ONLY the number.\n"
              f"Q: {question}\nExpected: {expected}\nAnswer: {response}")
    r = client.models.generate_content(model="gemini-2.5-flash", contents=prompt)
    try:    return float(r.text.strip().split()[0])
    except: return 0.5
# score fine-tuned vs base on the SAME held-out cases, average per metric, compare.

# Output: (illustrative - returns a 0.0-1.0 float per metric; needs GOOGLE_API_KEY set)
```

#### 💡 What just happened

⚡ What just happened? The base scored 0.0 and the fine-tune 1.0 on these held-out cases - a clear, positive delta, so this one ships. The mechanism is what matters: **same cases, both models, identical scoring, compare the delta** instead of trusting the loss curve. Note the SDK - the current judge uses `from google import genai; client.models.generate_content(model="gemini-2.5-flash")`, not the retired `google.generativeai`. And remember the cold-open: a beautiful loss curve would have told you none of this.

- Set base and fine-tuned scores per dimension
- Watch the deltas and the ship / do-not-ship verdict

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: The Benchmark Portfolio - Why MMLU Alone Lies

### The Benchmark Portfolio - Why MMLU Alone Lies

Classic benchmarks are saturated. Use a portfolio (MMLU-Pro, GPQA, IFEval, EvalPlus, LMArena) matched to your use case.

Standardized benchmarks let you compare against public baselines - but the famous ones are **saturated**: MMLU (~90%), GSM8K (94%+), and the original HumanEval (96%+) are all so easy for modern models that everyone clusters at the top and the benchmark stops discriminating. The 2026 portfolio: **MMLU-Pro** (10-option, a hygiene minimum), **GPQA Diamond** (graduate reasoning, the current gold standard), **IFEval** (verifiable instruction-following, objective), **EvalPlus** (code), and **LMArena** (formerly Chatbot Arena - human preference). No single one is enough.

> 📏 **Analogy**
>
> Judging a decathlete by the 100m alone is useless if every contender runs it in 10.0 - the event no longer separates them. You need the **whole decathlon** (a portfolio of events), and you especially want the events where times still spread out (GPQA), not the ones everyone maxes (MMLU).

Two strong 2026 models both score ~89% on MMLU. Which benchmark is most likely to actually separate them?

**📝 `02_portfolio.py`** - *runnable*

In [ ]:
# BENCHMARK PORTFOLIO - a benchmark only helps if it is NOT saturated AND separates the two models.
bench = {
    "MMLU":         {"a": 0.88, "b": 0.89, "saturated": True},
    "MMLU-Pro":     {"a": 0.72, "b": 0.79, "saturated": False},
    "GPQA-Diamond": {"a": 0.41, "b": 0.55, "saturated": False},
    "IFEval":       {"a": 0.68, "b": 0.83, "saturated": False},
}
print("  benchmark      A     B    gap   discriminates?")
for name, d in bench.items():
    gap = abs(d["a"] - d["b"])
    ok = (not d["saturated"]) and gap >= 0.03
    tag = "yes" if ok else ("SATURATED" if d["saturated"] else "too close")
    print(f"  {name:13s} {d['a']:.2f}  {d['b']:.2f}  {gap:.2f}   {tag}")

# Output:
#   benchmark      A     B    gap   discriminates?
#   MMLU          0.88  0.89  0.01   SATURATED
#   MMLU-Pro      0.72  0.79  0.07   yes
#   GPQA-Diamond  0.41  0.55  0.14   yes
#   IFEval        0.68  0.83  0.15   yes

- A benchmark is only useful if it is NOT saturated AND the gap between the two models is meaningful.
- MMLU is flagged SATURATED (gap 0.01) - it cannot separate the models.
- MMLU-Pro, GPQA-Diamond, and IFEval all show real gaps - they discriminate.
- The takeaway: report a portfolio, and always measure each model against the BASE, not just against each other.

#### 💡 What just happened

⚡ What just happened? MMLU, GSM8K, and the original HumanEval are **saturated** - they no longer tell strong models apart. The 2026 replacements are harder and still discriminate: **MMLU-Pro** (knowledge), **GPQA Diamond** (reasoning), **IFEval** (instruction-following), **EvalPlus** (code), and **LMArena**'s (formerly Chatbot Arena) millions of human votes (overall preference). Pick a portfolio that matches your deployment, and always compare against the base - instead of only against each other - to measure the actual fine-tuning gain.

- Pick a benchmark and a model tier
- See whether it still discriminates or is saturated

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: LLM-as-Judge - Cheap, Fast, and Biased

### LLM-as-Judge - Cheap, Fast, and Biased

An LLM judge is ~500-5000x cheaper than humans but has position, verbosity, and self-enhancement biases. Defend against them.

An LLM-as-judge agrees with human preference ~80-90% of the time at a tiny fraction of the cost of human annotation - which is why it is the default for scoring open-ended answers. But it is not neutral. It has three well-documented biases: **position** (it favors whichever answer is shown first), **verbosity** (it reads longer as better), and **self-enhancement** (it rates its own model family higher).

> 🥾 **Analogy**
>
> A judge who always gives the gymnast who performs FIRST a slightly higher score. The fix is not to find a perfect judge - it is to **run the routine twice in swapped order** and only trust a result that holds both ways, and to convene a **panel** of judges from different schools so no single bias dominates.

An LLM judge rates whichever answer is presented FIRST higher, regardless of content. What is the simplest defense?

**📝 `03_judge.py`** - *runnable*

In [ ]:
# LLM-AS-JUDGE DEFENSES - position swap (accept only if consistent) + jury majority.
def biased_judge(favors_first=True):
    return "first" if favors_first else "second"   # a judge that leans to slot 1

def swap_consistent(favors_first):
    v1 = biased_judge(favors_first)                # A in slot 1
    v2 = biased_judge(favors_first)                # A in slot 2 -> "first" now means B
    win1 = "A" if v1 == "first" else "B"
    win2 = "B" if v2 == "first" else "A"
    return win1 if win1 == win2 else "TIE (inconsistent -> position bias detected)"

from collections import Counter
def jury(votes):
    top, n = Counter(votes).most_common(1)[0]
    return top if n > len(votes)/2 else "ABSTAIN (no majority)"

print(f"  slot-1-biased judge, swapped -> {swap_consistent(True)}")
print(f"  jury [A, A, B]   -> {jury(['A','A','B'])}")
print(f"  jury [A, B, TIE] -> {jury(['A','B','TIE'])}")

# Output:
#   slot-1-biased judge, swapped -> TIE (inconsistent -> position bias detected)
#   jury [A, A, B]   -> A
#   jury [A, B, TIE] -> ABSTAIN (no majority)

- A slot-1-biased judge, run in swapped order, returns TIE - the inconsistency EXPOSES the position bias instead of hiding it.
- The `jury` takes a majority vote across judge families; [A, A, B] → A.
- With no majority ([A, B, TIE]) the jury ABSTAINS rather than guessing - an honest ‘we cannot tell’.
- Together: swap + jury + calibration on a labeled set removes most systematic bias.

#### 💡 What just happened

⚡ What just happened? LLM-as-judge is ~500-5000× cheaper than human evaluation but needs a **three-defense stack**: **position swap** (accept only if the verdict holds when you flip the order), **ensemble/jury** (multiple judge families, majority vote), and **calibration** on a small labeled set. For domain scoring, analytic rubrics with binary MET/UNMET criteria give the highest inter-rater reliability. Self-hosted judges (Prometheus, Glider) cut cost further for frequent evals.

---

## 🎯 Concept 4: The Safety Gate - Fine-Tuning Can Break Alignment

### The Safety Gate - Fine-Tuning Can Break Alignment

Fine-tuning on even benign data degrades safety. Gate the deploy on a safety suite + a forgetting check.

Here is the finding that surprises everyone: fine-tuning on *benign*, on-task data can silently strip a model's safety refusals, and a handful of adversarial examples can compromise alignment almost entirely (Qi et al. 2023). So the last gate before deploy is not accuracy - it is **safety regression** (does it still refuse harmful requests?) plus **catastrophic forgetting** (did it lose general ability?), measured by perplexity on held-out text. **Perplexity** is a lower-is-better score for how fluently a model predicts held-out text - roughly, how “surprised” it is; a sharp jump means it has forgotten general language ability.

> 🛡️ **Analogy**
>
> A new hire aces the job-skills test but, in the rush to train them on your product, nobody re-checked that they still follow the safety rules. You do not let them on the floor until they pass the **safety induction** again - every time, no exceptions.

You fine-tuned only on polite, benign customer-support chats. Could that harm the model's safety refusals?

**📝 `04_safety.py`** - *runnable*

In [ ]:
# SAFETY GATE - block the deploy on a safety-refusal drop OR catastrophic forgetting.
def deploy_gate(base_refusal, ft_refusal, base_ppl, ft_ppl):
    reasons = []
    if ft_refusal < 0.95:
        reasons.append(f"SAFETY: refusal {ft_refusal:.0%} < 95% (base {base_refusal:.0%}) - alignment degraded")
    if ft_ppl > base_ppl * 1.10:
        reasons.append(f"FORGETTING: perplexity {ft_ppl:.1f} > 1.1x base {base_ppl:.1f}")
    return ("BLOCK", reasons) if reasons else ("PASS", ["within thresholds"])

for label, args in [("benign fine-tune", (1.00, 0.62, 8.2, 8.4)),
                    ("safe fine-tune",   (1.00, 0.98, 8.2, 8.5))]:
    verdict, reasons = deploy_gate(*args)
    print(f"  {label:18s} -> {verdict}")
    for r in reasons: print(f"      - {r}")

# Output:
#   benign fine-tune   -> BLOCK
#       - SAFETY: refusal 62% < 95% (base 100%) - alignment degraded
#   safe fine-tune     -> PASS
#       - within thresholds

- The gate BLOCKS the ‘benign’ fine-tune: its refusal rate collapsed well below the 95% gate (the base refused everything) - alignment degraded even on harmless data.
- The ‘safe’ fine-tune PASSES: refusals held near the base and perplexity barely moved.
- Two thresholds: refusal rate must stay high (safety) AND perplexity must not jump (no catastrophic forgetting).
- This gate is a hard blocker - a model that fails it never reaches production, no matter how good its accuracy.

#### 💡 What just happened

⚡ What just happened? Safety regression is the **#1 deploy risk** for fine-tuned models. Run a suite - **HarmBench** (harmful behaviors), **ToxiGen** (implicit toxicity), **BBQ** (bias), **SORRY-Bench** (fine-grained refusal) - and check **catastrophic forgetting** via perplexity on WikiText-2/C4 (smoother than accuracy). LoRA's constrained update is itself a regularizer that preserves base knowledge, and replay-mixing roughly 5-10% general data helps (the tradeoff is a slightly larger training mix) - but you always run the gate.

- Drag refusal rate and perplexity for base vs fine-tuned
- See the gate PASS or BLOCK the deploy

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Merge the Adapter - LoRA Merge & Model Merging

### Merge the Adapter - LoRA Merge & Model Merging

Fold the adapter into the base for a standalone model, or blend whole models with TIES/DARE.

Once the model passes eval and safety, you fold the adapter in. **LoRA merge** (`merge_and_unload`) folds B·A back into W0 for one standalone model - zero adapter overhead, best for single-task deploy. You can also blend *whole* fine-tuned models with **mergekit**: **linear** (weighted average), **SLERP** (spherical interpolation of two models), **TIES** (trim redundant deltas, elect a sign, merge), and **DARE-TIES** (drop + rescale deltas, then TIES). The trick is handling *conflicting* updates.

> 💡 **Info**
>
> ⚠️ A QLoRA adapter cannot merge into a 4-bit baseIf you trained with QLoRA (Lesson 9.4), the base is 4-bit - `merge_and_unload()` cannot fold the adapter into it. Reload the base in fp16/bf16, merge, *then* re-quantize the merged model (Step 6). The tradeoff: merge-then-requantize is usually slightly worse than keeping the adapter separate at inference, so for a single task it is fine, but if quality is critical, compare both.

> 🍭 **Analogy**
>
> Two chefs each tweaked the same recipe. **Linear** averaging is dumping both sets of changes in equal measure - where they disagree (one added chili, one added sugar) you get a muddy half-and-half. **TIES** is a head chef who, on each disputed ingredient, picks the *direction* the majority favored and commits to it - no muddy compromise. That is why TIES-family merges beat plain averaging.

Two adapters change the same weight in OPPOSITE directions (+0.5 and -0.6). What does plain linear averaging do?

**📝 `05_merge.py`** - *runnable*

In [ ]:
# MERGE MATH - two fine-tunes as task vectors (deltas vs base). Merge 4 ways.
import math
t1 = [ 0.9, -0.2,  0.5,  0.05, -0.7]      # support-bot delta
t2 = [ 0.8,  0.3, -0.6,  0.04,  0.7]      # coding-bot delta

def linear(a, b, w=0.5):
    return [round(w*x + (1-w)*y, 3) for x, y in zip(a, b)]

def ties(a, b, keep=0.6):
    def trim(v):                              # 1) keep the top-|magnitude| fraction
        k = max(1, int(len(v)*keep)); thr = sorted((abs(x) for x in v), reverse=True)[k-1]
        return [x if abs(x) >= thr else 0.0 for x in v]
    ta, tb = trim(a), trim(b); out = []
    for x, y in zip(ta, tb):
        s = 1 if x + y >= 0 else -1           # 2) ELECT a sign
        vals = [v for v in (x, y) if v != 0 and (v > 0) == (s > 0)]   # 3) keep only agreeing deltas
        out.append(round(sum(vals)/len(vals), 3) if vals else 0.0)
    return out

def dare_ties(a, b, drop=0.5):
    def dare(v):                              # DARE: drop deltas, rescale survivors by 1/(1-drop)
        return [0.0 if i % 2 == 0 else round(x/(1-drop), 3) for i, x in enumerate(v)]
    return ties(dare(a), dare(b))             # fixed alternating mask here for reproducibility

print(f"  t1 (support): {t1}")
print(f"  t2 (coding):  {t2}")
print(f"  linear 50/50: {linear(t1, t2)}   <- position 3 muddied to -0.05 (interference)")
print(f"  TIES:         {ties(t1, t2)}   <- sign-elected: position 3 kept -0.6, no muddy average")
print(f"  DARE-TIES:    {dare_ties(t1, t2)}   <- dropped+rescaled deltas, then TIES")
print("  DARE-TIES is strongest for SFT+DPO merges; SLERP DEGRADES aligned models")

# Output:
#   t1 (support): [0.9, -0.2, 0.5, 0.05, -0.7]
#   t2 (coding):  [0.8, 0.3, -0.6, 0.04, 0.7]
#   linear 50/50: [0.85, 0.05, -0.05, 0.045, 0.0]   <- position 3 muddied to -0.05 (interference)
#   TIES:         [0.85, 0.0, -0.6, 0.0, 0.7]   <- sign-elected: position 3 kept -0.6, no muddy average
#   DARE-TIES:    [0.0, 0.6, 0.0, 0.09, 0.0]   <- dropped+rescaled deltas, then TIES

- Position 3: the two deltas conflict (+0.5 vs -0.6). **Linear** muddies them to -0.05 - interference that weakens both.
- **TIES** trims small deltas, then *elects a sign* per weight and keeps only the agreeing values - position 3 becomes -0.6, a committed direction.
- That sign-election is why TIES-family merges reduce interference between models.
- The next block shows the real one-liners (PEFT merge_and_unload, mergekit dare_ties).

**📝 `05b_merge_real.py PEFT +`** - *mergekit*

In [ ]:
# THE REAL MERGES (illustrative). LoRA merge = PEFT; whole-model merge = mergekit.
merge_lora = """
from peft import AutoPeftModelForCausalLM
model = AutoPeftModelForCausalLM.from_pretrained("./netsetos-lora", device_map="auto")
merged = model.merge_and_unload()          # fold B*A into W0 -> standalone model
merged.save_pretrained("./netsetos-merged")
"""
mergekit_yaml = """
# pip install mergekit ; mergekit-yaml config.yaml ./out
models:
  - model: netsetos/support-v1
  - model: netsetos/coding-v1
merge_method: dare_ties          # best for SFT+DPO-aligned models (SLERP degrades them)
base_model: google/gemma-3-4b-it # pick the current base
dtype: bfloat16
"""
print("LoRA merge -> merge_and_unload; whole-model -> mergekit (dare_ties). Re-run eval on the merged model.")

# Output:
# LoRA merge -> merge_and_unload; whole-model -> mergekit (dare_ties). Re-run eval on the merged model.

#### 💡 What just happened

⚡ What just happened? Two ways to merge: fold a **LoRA adapter** into its base (`merge_and_unload`) for a zero-overhead standalone model, or blend **whole models** with mergekit. Plain linear averaging muddies conflicting deltas; **TIES** trims + sign-elects to remove that interference, and **DARE-TIES** adds a drop-and-rescale step - the strongest choice for combining SFT+DPO-aligned models instead of plain averaging. **SLERP degrades aligned models**, so avoid it there. Always re-run eval + safety on the merged result.

- Pick linear / SLERP / TIES / DARE-TIES
- Watch conflicting deltas muddy or resolve

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: Quantize for the Target - GGUF, AWQ, FP8

### Quantize for the Target - GGUF, AWQ, FP8

Shrink the model to fit the deployment. The format and the kernel both matter.

A merged fp16 7B is ~14GB - too big for a laptop and wasteful on a GPU. Quantization shrinks it, and the right format depends on *where* it runs. **GGUF Q4_K_M / Q6_K** for local/Ollama (CPU-friendly), **AWQ 4-bit + the Marlin kernel** for GPU throughput, **FP8** on H100/H200. The subtle part: a quantization algorithm is only as fast as its *kernel* - AWQ without Marlin can actually be slower than fp16.

> 📦 **Analogy**
>
> Quantization is **vacuum-packing** luggage for a trip. Q6_K is a light pack (fits more, nothing missed). Q4_K_M is a tight pack (much smaller, a few wrinkles). But a great vacuum bag (AWQ) with no working pump (the Marlin kernel) saves you nothing - the tool and the technique both have to be there.

You quantize to AWQ 4-bit and it runs SLOWER than fp16. Most likely reason?

**📝 `06_quant.py`** - *runnable*

In [ ]:
# QUANTIZE FOR THE TARGET - size, quality, and where each format belongs (7B model).
def quant(fmt, B):
    table = {
        "fp16":       (B*2.0,  1.00, "baseline"),
        "GGUF Q6_K":  (B*0.79, 0.98, "local, quality-sensitive"),
        "GGUF Q4_K_M":(B*0.63, 0.94, "local / Ollama, VRAM-tight"),
        "AWQ+Marlin": (B*0.6,  0.96, "GPU throughput (needs the Marlin kernel)"),
        "FP8 W8A8":   (B*1.0,  0.99, "H100/H200 only"),
    }
    return table[fmt]

print(f"  {'format':12s} {'VRAM':>7} {'quality':>8}   note")
for f in ("fp16", "GGUF Q6_K", "GGUF Q4_K_M", "AWQ+Marlin", "FP8 W8A8"):
    gb, q, note = quant(f, 7)
    print(f"  {f:12s} {gb:6.1f}GB {q*100:6.0f}%   {note}")
print("  kernels > algorithms: AWQ WITHOUT Marlin can be slower than fp16; AutoAWQ archived -> llm-compressor")

# Output:
#   format          VRAM  quality   note
#   fp16           14.0GB    100%   baseline
#   GGUF Q6_K       5.5GB     98%   local, quality-sensitive
#   GGUF Q4_K_M     4.4GB     94%   local / Ollama, VRAM-tight
#   AWQ+Marlin      4.2GB     96%   GPU throughput (needs the Marlin kernel)
#   FP8 W8A8        7.0GB     99%   H100/H200 only

- Q6_K keeps ~98% quality at well under half of fp16's memory; Q4_K_M is smaller still (~94%).
- AWQ+Marlin and Q4_K_M land at similar VRAM, but AWQ targets GPU throughput (with the kernel) while Q4_K_M targets local/CPU.
- FP8 keeps ~99% quality but only helps on H100/H200 hardware.
- The ‘quality’ column is roughly benchmark/perplexity retention vs the fp16 baseline (Step 4’s perplexity metric) - a ballpark, not a promise; measure it on YOUR eval.
- The one-liner to remember: kernels matter more than the algorithm, and AutoAWQ is archived - use llm-compressor.

#### 💡 What just happened

⚡ What just happened? Match the format to the target: **GGUF Q4_K_M/Q6_K** for local/Ollama, **AWQ+Marlin** for GPU throughput, **FP8** for H100/H200. The counter-intuitive lesson: **kernels beat algorithms** - AWQ without Marlin can be slower than fp16. Tooling moved too: **AutoAWQ is archived; use llm-compressor**. Unsloth's `save_pretrained_gguf` does merge+convert+quantize in one call.

- Choose a format and model size
- Compare VRAM, quality, and throughput

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: Deploy - Merge vs Multi-LoRA, Ollama vs vLLM

### Deploy - Merge vs Multi-LoRA, Ollama vs vLLM

Pick the target by task count, traffic, and whether it runs at the edge. Serving ops are Modules 12-14.

Finally you ship it. The decision is small: **one task at the edge** → merge → GGUF → **Ollama** (CPU-friendly, local). **One task on a GPU** → merge → **vLLM** (OpenAI-compatible, continuous batching). **Many tasks** → keep the adapters separate and serve **vLLM multi-LoRA** (one base + N adapters, routed per request). **Bursty/managed** → HF scale-to-zero or Vertex. The deep serving ops - SLO metrics, KV-cache routing, canary/blue-green, autoscaling - are Modules 12-14.

> 🛣️ **Analogy**
>
> Choosing a vehicle for the trip. One passenger, back roads → a motorbike (Ollama at the edge). One passenger, highway → a car (vLLM). Several passengers going to different stops → one bus with a route (multi-LoRA), not five separate cars. You match the vehicle to the trip, not the other way around.

You have 3 task adapters and want one GPU serving all three with independent updates. Best deploy?

**📝 `07_deploy.py`** - *runnable*

In [ ]:
# DEPLOY DECISION - by task count, traffic, and whether it runs at the edge.
def deploy(n_tasks, traffic, edge):
    if edge:
        return ("merge -> GGUF -> Ollama", "single model, CPU-friendly, runs local/edge")
    if n_tasks == 1:
        return ("merge_and_unload -> vLLM", "one standalone model, zero adapter overhead")
    if n_tasks > 1 and traffic != "high":
        return ("vLLM multi-LoRA (--enable-lora)", f"one base + {n_tasks} adapters routed per request")
    return ("managed (HF scale-to-zero / Vertex) or dedicated vLLM", "bursty/high; ops in Modules 12-14")

for label, args in [("1 task, edge", (1,"low",True)), ("1 task, GPU", (1,"med",False)),
                    ("4 tasks, GPU", (4,"med",False)), ("bursty many", (6,"high",False))]:
    tgt, why = deploy(*args)
    print(f"  {label:14s} -> {tgt}")

# Output:
#   1 task, edge   -> merge -> GGUF -> Ollama
#   1 task, GPU    -> merge_and_unload -> vLLM
#   4 tasks, GPU   -> vLLM multi-LoRA (--enable-lora)
#   bursty many    -> managed (HF scale-to-zero / Vertex) or dedicated vLLM

- Edge/local single task → merge to GGUF and run on Ollama (CPU-friendly).
- Single task on a GPU → merge_and_unload and serve on vLLM (zero adapter overhead).
- Several tasks → vLLM multi-LoRA: one base + N adapters routed per request (the memory win from 9.5).
- Bursty/high traffic → managed or dedicated vLLM - and the SLO/scaling depth is Modules 12-14.
- Constraint: multi-LoRA co-serves adapters only if they share the SAME base model (and compatible rank/target-modules) - adapters on different bases need separate deployments.

#### 💡 What just happened

⚡ What just happened? The deploy decision follows task count + traffic + edge: **merge→Ollama** for a single local model, **vLLM** for GPU serving, **vLLM multi-LoRA** for many tasks on one base, and **managed** (HF scale-to-zero / Vertex) for bursty traffic. That is the shipping decision this lesson owns; the production ops around it - monitoring, autoscaling, canary, drift, cost - are **Modules 12-14**.

- Set task count and traffic
- Get Ollama / vLLM / multi-LoRA / managed

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 8: The India Deploy Path - Sarvam, IndiaAI, Bhashini, DPDP

### The India Deploy Path - Sarvam, IndiaAI, Bhashini, DPDP

Cheap Indian GPUs, sovereign models, and data-residency compliance for regulated sectors.

For Indian deployments the stack is concrete and cheap. Compute: the **IndiaAI Mission** offers subsidized H100s (reported around Rs 65/GPU-hr - published rates move, so treat every rupee figure here as indicative), and **E2E / Jarvislabs** H100s run several times cheaper than AWS with minute billing. Models: Sarvam ships open, Indic-multilingual sovereign models - **Sarvam-M (24B)** today, with larger releases arriving; check Sarvam's current model catalog for exact sizes before you commit. Multilingual glue: **AI4Bharat Bhashini** APIs (speech-to-text, text-to-speech, translation across 22 languages). And **DPDP Act** data-residency makes Indian clouds the natural pick for regulated sectors.

> 🇮🇳 **Analogy**
>
> Building for India is like sourcing ingredients locally: the produce (compute) is cheaper at the local market (IndiaAI/E2E) than imported (AWS), the local varieties (Sarvam, Bhashini) are tuned to local tastes (22 languages), and the food-safety rules (DPDP) are written for local kitchens - so cooking at home is both cheaper and compliant.

A regulated Indian bank must keep training and inference data in-country. Which deploy path fits best?

**📝 `08_india.py`** - *runnable*

In [ ]:
# INDIA DEPLOY PLANNER - budget + languages -> infra + model + Bhashini.
def india(budget_inr_month, langs):
    extra = "Bhashini APIs (STT/TTS/translate, 22 languages)" if langs > 1 else "single-language endpoint"
    if budget_inr_month < 15000:
        return ("HF scale-to-zero (T4) or Jarvislabs spot", "Sarvam-1 / Gemma 3, GGUF+Ollama", extra)
    if budget_inr_month <= 40000:
        return ("E2E / Jarvislabs H100 (~Rs 225-249/hr, peak hours only)", "Sarvam-M (24B) AWQ on vLLM", extra)
    return ("IndiaAI Mission H100 (~Rs 65/hr, subsidized)", "Sarvam-M / larger open model, vLLM multi-LoRA", extra)

for b, l in [(9000, 1), (30000, 3), (120000, 5)]:
    infra, model, extra = india(b, l)
    print(f"  budget Rs {b:<7}/mo langs={l} -> {infra}")
    print(f"      model: {model}")
print("  DPDP data-residency: Indian clouds comply by default - the natural pick for regulated sectors")

# Output:
#   budget Rs 9000   /mo langs=1 -> HF scale-to-zero (T4) or Jarvislabs spot
#       model: Sarvam-1 / Gemma 3, GGUF+Ollama
#   budget Rs 30000  /mo langs=3 -> E2E / Jarvislabs H100 (~Rs 225-249/hr, peak hours only)
#       model: Sarvam-M (24B) AWQ on vLLM
#   budget Rs 120000 /mo langs=5 -> IndiaAI Mission H100 (~Rs 65/hr, subsidized)
#       model: Sarvam-M / larger open model, vLLM multi-LoRA

- Low budget / single language → scale-to-zero T4 or Jarvislabs spot with a small Sarvam/Gemma model on Ollama.
- Mid budget / several languages → E2E or Jarvislabs H100 during peak hours, Sarvam-M on vLLM, plus Bhashini for speech/translation.
- High budget / many languages → subsidized IndiaAI H100s with a larger Sarvam / open model on vLLM multi-LoRA.
- Across all tiers, Indian clouds satisfy DPDP data-residency by default. The rupee GPU rates shown are indicative and move over time - re-quote before you budget.

#### 💡 What just happened

⚡ What just happened? The India path is cheap and sovereign: **IndiaAI Mission** (~Rs 65/GPU-hr, indicative) and **E2E/Jarvislabs** for compute, **Sarvam** (Sarvam-M 24B and larger open releases) for models, **Bhashini** for 22-language speech/translation, and **DPDP** data-residency handled by default on Indian clouds. For regulated sectors (banking, health, government), that residency guarantee is often the deciding factor, not price.

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole ship-it pipeline
> Training loss lied; a held-out **A/B against the base** tells the truth (Step 1), read through a **benchmark portfolio** because MMLU is saturated (Step 2), scored by an **LLM judge** whose position/verbosity/self-enhancement biases you defend against (Step 3). Before shipping you clear the **safety gate** - fine-tuning breaks alignment even on benign data (Step 4). Then you **merge** the adapter (linear muddies conflicts; TIES/DARE-TIES resolve them, Step 5), **quantize** to the target where kernels beat algorithms (Step 6), and pick a **deploy target** by task count and traffic (Step 7) - with a cheap, sovereign **India path** when you need it (Step 8).

> 📦 **Info**
>
> ➡️ Where this goes nextThis lesson closes Module 9 - you can now fine-tune, evaluate, merge, quantize, and ship. **Module 12 covers** production deployment depth (SLO metrics, autoscaling, KV-cache routing). We revisit cost and latency engineering in Module 13, and eval-driven monitoring, drift, and canary rollout in Module 14. The reference tools live in [mergekit](https://github.com/arcee-ai/mergekit) and [llm-compressor](https://github.com/vllm-project/llm-compressor).

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Eval, Merge& Deploy**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-9_6.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-9.6.html` - regenerate this notebook whenever the source HTML is updated.*
